In [47]:
import random
from qdrant_client import QdrantClient
import instructor
import json
from pydantic import BaseModel, Field
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langsmith import Client
import tiktoken

In [48]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [49]:
qdrant_client = QdrantClient(url="http://localhost:6333")

### Load all data from Qdrant

In [50]:
all_points = []
next_offset = None
batch_size = 100
collection_name = 'Amazon-shopping-collection-01-hybrid-search'

while True:
    points, next_offset = qdrant_client.scroll(
        collection_name = collection_name,
        limit = batch_size,
        offset = next_offset,
        with_payload = True,
        with_vectors = False
    )
    print(f">>>>>>>>>>>>>>>>>>>>>>> next_offset: {next_offset}")
    all_points.extend(points)
    if next_offset is None:
        break


>>>>>>>>>>>>>>>>>>>>>>> next_offset: 101
>>>>>>>>>>>>>>>>>>>>>>> next_offset: 201
>>>>>>>>>>>>>>>>>>>>>>> next_offset: 301
>>>>>>>>>>>>>>>>>>>>>>> next_offset: 401
>>>>>>>>>>>>>>>>>>>>>>> next_offset: 501
>>>>>>>>>>>>>>>>>>>>>>> next_offset: 601
>>>>>>>>>>>>>>>>>>>>>>> next_offset: 701
>>>>>>>>>>>>>>>>>>>>>>> next_offset: 801
>>>>>>>>>>>>>>>>>>>>>>> next_offset: 901
>>>>>>>>>>>>>>>>>>>>>>> next_offset: None


In [51]:
(all_points)

[Record(id=1, payload={'preprocessed_description': 'FENIFOX Bluetooth Mouse, Slim Mini Portable Flat Travel Wireless Mouse Rechargeable Quiet Ultra-Thin Mice Compatible with Laptop,Tablet,Notebook,PC (Silver and White)【LIGHT and ULTRA-THIN】: Slim and light design, only 0.6 in thick, 0.1 lbs weight. Made of ABS+PC material, it doesn’t occupy USB port when used. The lightweight and portable designing is the first choice for travel. 【LOW DECIBEL BUTTON】: Low decibel left and right mouse buttons, Suitable for libraries, conference rooms, coffee shops and other occasions. 【ENERGE SAVING and ENVIRONMENT-FRIENDLY】: The mouse has its own power switch and enter sleep mode automatically. It is the most safe and stable lithium polymer battery, work over 3 weeks after one fully charge , can be charged more than hundred times. 【WIDE COMPATIBILITY】: Compatible with the devices,that have 3.0 Version Bluetooth or higher.Like PC,laptop,tablet.But Not compatible with iphone or ipad. 【EASY to OPERATE】: T

### Split the data in 2 groups:  50 items for synthetic question generation, 95 items to evaluate their relevance for generated sythentic questions

In [52]:
rng = random.Random(42)
print(rng)
indices = list(range(len(all_points)))
indices

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [53]:
rng.shuffle(indices)
indices

[776,
 507,
 895,
 922,
 33,
 483,
 85,
 750,
 354,
 523,
 184,
 809,
 418,
 615,
 682,
 501,
 760,
 49,
 732,
 336,
 450,
 790,
 350,
 467,
 622,
 476,
 614,
 554,
 365,
 770,
 630,
 940,
 797,
 496,
 924,
 369,
 596,
 720,
 53,
 426,
 56,
 264,
 772,
 185,
 466,
 789,
 174,
 150,
 741,
 113,
 428,
 965,
 18,
 189,
 414,
 589,
 666,
 763,
 221,
 586,
 581,
 299,
 638,
 386,
 364,
 879,
 334,
 16,
 202,
 1,
 38,
 290,
 493,
 247,
 506,
 896,
 713,
 891,
 76,
 911,
 792,
 508,
 177,
 842,
 200,
 765,
 909,
 293,
 538,
 24,
 694,
 210,
 424,
 528,
 34,
 928,
 398,
 607,
 298,
 848,
 668,
 282,
 939,
 944,
 606,
 959,
 934,
 731,
 287,
 527,
 807,
 229,
 436,
 28,
 374,
 819,
 316,
 560,
 241,
 149,
 516,
 417,
 660,
 198,
 585,
 325,
 823,
 702,
 949,
 515,
 864,
 836,
 877,
 359,
 594,
 788,
 188,
 66,
 355,
 799,
 902,
 920,
 942,
 330,
 727,
 396,
 463,
 393,
 311,
 680,
 744,
 921,
 461,
 522,
 631,
 490,
 195,
 256,
 693,
 740,
 577,
 157,
 612,
 392,
 969,
 688,
 361,
 816,
 401,
 

In [54]:
sample_indx = set(indices[:50])
sample_indx

{33,
 49,
 53,
 56,
 85,
 113,
 150,
 174,
 184,
 185,
 264,
 336,
 350,
 354,
 365,
 369,
 418,
 426,
 450,
 466,
 467,
 476,
 483,
 496,
 501,
 507,
 523,
 554,
 596,
 614,
 615,
 622,
 630,
 682,
 720,
 732,
 741,
 750,
 760,
 770,
 772,
 776,
 789,
 790,
 797,
 809,
 895,
 922,
 924,
 940}

In [55]:
sample_50 = [p for i, p in enumerate(all_points) if i in sample_indx]
sample_50

[Record(id=34, payload={'preprocessed_description': 'Bose QuietComfort 25 Headphones Inline Mic/Remote Cable for Samsung & Android Devices (Blue)', 'image': 'https://m.media-amazon.com/images/I/31Pm7T7tscL.jpg', 'rating_number': 2284, 'price': None, 'average_rating': 4.6, 'parent_asin': 'B00XLIG4GW'}, vector=None, shard_key=None, order_value=None),
 Record(id=50, payload={'preprocessed_description': "[Works with Solar 3 Only] Foxpark Solar Wireless Backup Camera for Car, 1080P Back Up Camera with 3350mAh Rechargeable Battery, 70 Degree View Angle Adjustable, IP69K Waterproof Reverse Camera【Works with Solar 3 Only】This is an optional camera for Foxpark Solar 3 Backup Camera System (Note. Does not work with other backup camera systems and phones). You can install this optional camera for the Solar 3 monitor to tow your trailer/RV/boat. Even use it to monitor your baby/ puppy etc. The powerful wireless transmission distance meets most cars, such as sedans, pickups, trucks, vans, and RVs (

In [56]:
remaining_points = [p for i, p in enumerate(all_points) if i not in sample_indx]
len(remaining_points)

950

### Generate synthetic  questions

In [57]:
all_contxt_sample_50 = [ {"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in sample_50]
all_contxt_sample_50

[{'id': 'B00XLIG4GW',
  'text': 'Bose QuietComfort 25 Headphones Inline Mic/Remote Cable for Samsung & Android Devices (Blue)'},
 {'id': 'B0BSLCXVY4',
  'text': "[Works with Solar 3 Only] Foxpark Solar Wireless Backup Camera for Car, 1080P Back Up Camera with 3350mAh Rechargeable Battery, 70 Degree View Angle Adjustable, IP69K Waterproof Reverse Camera【Works with Solar 3 Only】This is an optional camera for Foxpark Solar 3 Backup Camera System (Note. Does not work with other backup camera systems and phones). You can install this optional camera for the Solar 3 monitor to tow your trailer/RV/boat. Even use it to monitor your baby/ puppy etc. The powerful wireless transmission distance meets most cars, such as sedans, pickups, trucks, vans, and RVs (Less than 33 ft). 【3 Mins DIY Wireless Installation】Zero wiring, zero drilling. Powered by a built-in battery and sunshine. No need to wire to the reverse light. You don't need to pay $200 for professional installation. Easy install Solar 3 w

In [58]:
all_contxt_remaining_points = [ {"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in remaining_points]
all_contxt_remaining_points

[{'id': 'B06ZZJ6C3P',
  'text': 'FENIFOX Bluetooth Mouse, Slim Mini Portable Flat Travel Wireless Mouse Rechargeable Quiet Ultra-Thin Mice Compatible with Laptop,Tablet,Notebook,PC (Silver and White)【LIGHT and ULTRA-THIN】: Slim and light design, only 0.6 in thick, 0.1 lbs weight. Made of ABS+PC material, it doesn’t occupy USB port when used. The lightweight and portable designing is the first choice for travel. 【LOW DECIBEL BUTTON】: Low decibel left and right mouse buttons, Suitable for libraries, conference rooms, coffee shops and other occasions. 【ENERGE SAVING and ENVIRONMENT-FRIENDLY】: The mouse has its own power switch and enter sleep mode automatically. It is the most safe and stable lithium polymer battery, work over 3 weeks after one fully charge , can be charged more than hundred times. 【WIDE COMPATIBILITY】: Compatible with the devices,that have 3.0 Version Bluetooth or higher.Like PC,laptop,tablet.But Not compatible with iphone or ipad. 【EASY to OPERATE】: There just need thre

In [59]:
SYSTEM_PROMPT = f"""
You are a test-data generator for a Retrieval-Augmented Generation (RAG) system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions about the products in stock by retrieving the most relevant products and grounding its answers in them.

## Your input
You will be given a sample of 50 products, each as an object with an `id` and a `text` description. This sample is part of a larger collection that will eventually contain 1000 products.

## Your task
Generate exactly 30 questions that a real customer of this e-shop might ask, split into three categories:

1. **Single-product (15 questions)** — answerable using exactly ONE product.
2. **Multi-product (10 questions)** — answerable using 2 to 3 products. Never more than 3.
3. **Unanswerable (5 questions)** — plausible customer questions that CANNOT be answered from any of the provided products.

## Constraints
- Write questions from the customer's point of view, in natural language.
- Refer to the items as "products". Never use the word "chunk".
- Keep questions specific. Even in the full 1000-product collection, a good question should be answerable from at most 4 products. Avoid broad or generic questions that a large number of products could satisfy.

## Products
{json.dumps(all_contxt_sample_50, indent=2)}
"""

In [60]:
print(SYSTEM_PROMPT)


You are a test-data generator for a Retrieval-Augmented Generation (RAG) system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions about the products in stock by retrieving the most relevant products and grounding its answers in them.

## Your input
You will be given a sample of 50 products, each as an object with an `id` and a `text` description. This sample is part of a larger collection that will eventually contain 1000 products.

## Your task
Generate exactly 30 questions that a real customer of this e-shop might ask, split into three categories:

1. **Single-product (15 questions)** — answerable using exactly ONE product.
2. **Multi-product (10 questions)** — answerable using 2 to 3 products. Never more than 3.
3. **Unanswerable (5 questions)** — plausible customer questions that CANNOT be answered from any of the provided products.

## Constraints
- Write questions from the customer's point of view, in natural language.
-

In [61]:
instructor_client = instructor.from_provider("openai/gpt-5.4", mode=instructor.Mode.RESPONSES_TOOLS)


In [62]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            }
        },
    },
}

In [63]:
class SyntheticQuestion(BaseModel):
    reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
    question: str = Field(description="Suggested question.")
    chunk_ids: list[str] = Field(description="ID of the chunk that could be used to answer the question.")
    answer_example: str = Field(description="Suggested answer grounded in the context.")

class SyntheticQuestions(BaseModel):
    synthetic_questions: list[SyntheticQuestion] = Field(description="List of synthetic questions")

In [65]:
response, raw_response = instructor_client.create_with_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT}
    ],
    reasoning={"effort": "none"},
    response_model=SyntheticQuestions
)
response

SyntheticQuestions(synthetic_questions=[SyntheticQuestion(reasoning='The Sony 75-inch TV description explicitly mentions Dolby Vision HDR, native 120Hz refresh rate, Google TV, and HDMI 2.1 gaming features, which supports a specific customer question about those capabilities.', question='Do you have a 75-inch Sony TV with Dolby Vision and a native 120Hz refresh rate for PS5 gaming?', chunk_ids=['B0B5LRMSQX'], answer_example='Yes — the Sony 75 Inch 4K Ultra HD TV X85K Series (product B0B5LRMSQX) is a 75-inch LED Smart Google TV with Dolby Vision HDR, native 120Hz refresh rate, and HDMI 2.1 gaming features like 4K/120, VRR, and ALLM.'), SyntheticQuestion(reasoning='The Foxpark backup camera product includes solar power, 1080P resolution, IP69K waterproofing, and a restriction that it works only with Solar 3, making it answerable by one product.', question='I need a wireless backup camera for my trailer that charges with solar power — do you have one that works with the Foxpark Solar 3 sy

In [66]:
response.synthetic_questions

[SyntheticQuestion(reasoning='The Sony 75-inch TV description explicitly mentions Dolby Vision HDR, native 120Hz refresh rate, Google TV, and HDMI 2.1 gaming features, which supports a specific customer question about those capabilities.', question='Do you have a 75-inch Sony TV with Dolby Vision and a native 120Hz refresh rate for PS5 gaming?', chunk_ids=['B0B5LRMSQX'], answer_example='Yes — the Sony 75 Inch 4K Ultra HD TV X85K Series (product B0B5LRMSQX) is a 75-inch LED Smart Google TV with Dolby Vision HDR, native 120Hz refresh rate, and HDMI 2.1 gaming features like 4K/120, VRR, and ALLM.'),
 SyntheticQuestion(reasoning='The Foxpark backup camera product includes solar power, 1080P resolution, IP69K waterproofing, and a restriction that it works only with Solar 3, making it answerable by one product.', question='I need a wireless backup camera for my trailer that charges with solar power — do you have one that works with the Foxpark Solar 3 system?', chunk_ids=['B0BSLCXVY4'], answ

In [67]:
for s_question in response.synthetic_questions:
    print(s_question.chunk_ids)

['B0B5LRMSQX']
['B0BSLCXVY4']
['B07ZM19Y9H']
['B091JVYV8F']
['B08H2323M6']
['B01F9RGSFE']
['B078P57ZWL']
['B09F9KF1L8']
['B09DBRM3R5']
['B07P85CDHG']
['B07PWTYHPL']
['B09M3P2W7M']
['B006C1ILUC']
['B08DXTLCBC']
['B091J2D4TG']
['B078XQ9LND', 'B07P85CDHG']
['B07J3X1DPG', 'B07J6PW4SH']
['B08H2323M6', 'B08DXTLCBC']
['B07WQFSPR1', 'B07BKPQMQL']
['B00XLIG4GW', 'B091JVYV8F']
['B08D69CDSX', 'B01F9RGSFE']
['B07GDJ3662', 'B09P197SM7']
['B0798JJ2T5', 'B0B5ZP46BJ']
['B0B681R5C7', 'B08QD3RXSN']
['B07ZM19Y9H', 'B0BGVZZH6F']
[]
[]
[]
[]
[]


In [68]:
synthetic_questions_relevant_data = [{"question_id": i, "question": item.question} for i, item in enumerate(response.synthetic_questions)]

In [69]:
synthetic_questions_relevant_data

[{'question_id': 0,
  'question': 'Do you have a 75-inch Sony TV with Dolby Vision and a native 120Hz refresh rate for PS5 gaming?'},
 {'question_id': 1,
  'question': 'I need a wireless backup camera for my trailer that charges with solar power — do you have one that works with the Foxpark Solar 3 system?'},
 {'question_id': 2,
  'question': 'Is there a body camera with built-in storage, night vision, and about 4 hours of recording time?'},
 {'question_id': 3,
  'question': 'Do you sell lightweight open-ear Bluetooth headphones for running that let me stay aware of my surroundings?'},
 {'question_id': 4,
  'question': 'Do you have a Wi‑Fi 6 PCIe card for a desktop PC with Bluetooth 5.0 and WPA3 support?'},
 {'question_id': 5,
  'question': "I'm looking for a 6-foot MFi-certified Lightning to USB-A cable for my iPhone. Do you have one?"},
 {'question_id': 6,
  'question': "Do you have a 27-inch curved gaming monitor that's 1440p and 144Hz with FreeSync?"},
 {'question_id': 7,
  'questi

In [70]:
SYSTEM_PROMPT_950 = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}

## Products
{json.dumps(all_contxt_remaining_points[0:50], indent=2)}
"""

In [71]:
print(SYSTEM_PROMPT_950)


You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
[
  {
    "question_id": 0,
    "question": "Do you have a 75-inch Sony TV with Dolby Vision and a native 120Hz refresh rate for PS5 gaming?"
  },
  {
    "questio

In [72]:
SYSTEM_PROMPT_STATIC_PART = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions_relevant_data, indent=2)}

## Products
"""

### count number of tokens...as token count should be greater than 1024 for prompt caching

In [73]:
def count_tokens(text):

    encoding = tiktoken.get_encoding("o200k_base")

    return len(encoding.encode(text))

In [74]:
print(count_tokens(SYSTEM_PROMPT_STATIC_PART))

1343


In [84]:
instructor_client = instructor.from_provider(
    "openai/gpt-5.4",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [76]:
class SyntheticQuestionMatchingChunk(BaseModel):
    reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
    question_id: int = Field(description="ID of the question.")
    chunk_ids: list[str] = Field(description="IDs of the chunks that could be used to answer the question.")

class SyntheticQuestionMatchingChunks(BaseModel):
    synthetic_question_matching_chunks: list[SyntheticQuestionMatchingChunk] = Field(description="List of synthetic questions and their matching chunks")

In [85]:
tmp_response, raw_response = instructor_client.create_with_completion(
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT_950}
    ],
    reasoning={"effort": "none"},
    response_model=SyntheticQuestionMatchingChunks
)

In [86]:
tmp_response.synthetic_question_matching_chunks

[SyntheticQuestionMatchingChunk(reasoning='Q0 asks for a 75-inch Sony TV with Dolby Vision and native 120Hz; none of the products are Sony 75-inch TVs, so no single product answers it.', question_id=0, chunk_ids=[]),
 SyntheticQuestionMatchingChunk(reasoning='Q1 asks for a wireless solar backup camera compatible with Foxpark Solar 3; no such product is present.', question_id=1, chunk_ids=[]),
 SyntheticQuestionMatchingChunk(reasoning='Q2 asks for a body camera with built-in storage, night vision, and ~4 hours recording; no body camera product matches.', question_id=2, chunk_ids=[]),
 SyntheticQuestionMatchingChunk(reasoning='Q3 asks for lightweight open-ear Bluetooth headphones for running; no open-ear headphone product matches.', question_id=3, chunk_ids=[]),
 SyntheticQuestionMatchingChunk(reasoning='Q4 asks for a Wi‑Fi 6 PCIe desktop card with Bluetooth 5.0 and WPA3; no such adapter is present.', question_id=4, chunk_ids=[]),
 SyntheticQuestionMatchingChunk(reasoning='Q5 asks for a 

In [87]:
for data in tmp_response.synthetic_question_matching_chunks:
    print(f"Question ID: {data.question_id}, Chunk IDs: {data.chunk_ids}")

Question ID: 0, Chunk IDs: []
Question ID: 1, Chunk IDs: []
Question ID: 2, Chunk IDs: []
Question ID: 3, Chunk IDs: []
Question ID: 4, Chunk IDs: []
Question ID: 5, Chunk IDs: []
Question ID: 6, Chunk IDs: []
Question ID: 7, Chunk IDs: []
Question ID: 8, Chunk IDs: []
Question ID: 9, Chunk IDs: []
Question ID: 10, Chunk IDs: ['B07ZVHBY7Z']
Question ID: 11, Chunk IDs: []
Question ID: 12, Chunk IDs: []
Question ID: 13, Chunk IDs: []
Question ID: 14, Chunk IDs: []
Question ID: 15, Chunk IDs: []
Question ID: 16, Chunk IDs: []
Question ID: 17, Chunk IDs: []
Question ID: 18, Chunk IDs: ['B013R9SBOC']
Question ID: 19, Chunk IDs: ['B06XPFXF8B']
Question ID: 20, Chunk IDs: []
Question ID: 21, Chunk IDs: []
Question ID: 22, Chunk IDs: []
Question ID: 23, Chunk IDs: []
Question ID: 24, Chunk IDs: []
Question ID: 25, Chunk IDs: []
Question ID: 26, Chunk IDs: []
Question ID: 27, Chunk IDs: []
Question ID: 28, Chunk IDs: []
Question ID: 29, Chunk IDs: []


In [43]:
raw_response.usage

ResponseUsage(input_tokens=13997, input_tokens_details=InputTokensDetails(cached_tokens=13696), output_tokens=1124, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=15121)

In [44]:
def get_relevant_chunks(synthetic_questions, remaining_points_batch):

    SYSTEM_PROMPT_950 = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions, indent=2)}

## Products
{json.dumps(remaining_points_batch, indent=2)}
"""

    response, raw_response = instructor_client.create_with_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_950}
        ],
        reasoning={"effort": "none"},
        response_model=SyntheticQuestionMatchingChunks
    )

    return response.synthetic_question_matching_chunks

In [88]:
answer_extended_chunks = []

for i in range(0, len(all_contxt_remaining_points), 50):

    print(f"Working on batch {(i+50)/50}")

    answer_extended_chunks.append(get_relevant_chunks(synthetic_questions_relevant_data, all_contxt_remaining_points[i:i+50]))

Working on batch 1.0
Working on batch 2.0
Working on batch 3.0
Working on batch 4.0
Working on batch 5.0
Working on batch 6.0
Working on batch 7.0
Working on batch 8.0
Working on batch 9.0
Working on batch 10.0
Working on batch 11.0
Working on batch 12.0
Working on batch 13.0
Working on batch 14.0
Working on batch 15.0
Working on batch 16.0
Working on batch 17.0
Working on batch 18.0
Working on batch 19.0


In [89]:
answer_extended_chunks

[[SyntheticQuestionMatchingChunk(reasoning='Q0 asks for a 75-inch Sony TV with Dolby Vision and native 120Hz. The only TV product is a 40-inch TCL 60Hz Roku TV, so no product alone answers it.', question_id=0, chunk_ids=[]),
  SyntheticQuestionMatchingChunk(reasoning='Q1 asks for a wireless solar backup camera for trailer compatible with Foxpark Solar 3. No such camera product exists.', question_id=1, chunk_ids=[]),
  SyntheticQuestionMatchingChunk(reasoning='Q2 asks for a body camera with built-in storage, night vision, and about 4 hours recording. No body camera product is present.', question_id=2, chunk_ids=[]),
  SyntheticQuestionMatchingChunk(reasoning='Q3 asks for lightweight open-ear Bluetooth headphones for running with situational awareness. No open-ear headphone product is present.', question_id=3, chunk_ids=[]),
  SyntheticQuestionMatchingChunk(reasoning='Q4 asks for a Wi‑Fi 6 PCIe desktop card with Bluetooth 5.0 and WPA3. No such adapter is present.', question_id=4, chunk_i

In [90]:
for i, data in enumerate(answer_extended_chunks):
    print(f"Batch {i+1}")
    for item in data:
        print(f"Question ID: {item.question_id}, Chunk IDs: {item.chunk_ids}")

Batch 1
Question ID: 0, Chunk IDs: []
Question ID: 1, Chunk IDs: []
Question ID: 2, Chunk IDs: []
Question ID: 3, Chunk IDs: []
Question ID: 4, Chunk IDs: []
Question ID: 5, Chunk IDs: []
Question ID: 6, Chunk IDs: []
Question ID: 7, Chunk IDs: []
Question ID: 8, Chunk IDs: []
Question ID: 9, Chunk IDs: []
Question ID: 10, Chunk IDs: ['B07ZVHBY7Z']
Question ID: 11, Chunk IDs: []
Question ID: 12, Chunk IDs: []
Question ID: 13, Chunk IDs: []
Question ID: 14, Chunk IDs: []
Question ID: 15, Chunk IDs: []
Question ID: 16, Chunk IDs: []
Question ID: 17, Chunk IDs: []
Question ID: 18, Chunk IDs: ['B013R9SBOC']
Question ID: 19, Chunk IDs: ['B06XPFXF8B']
Question ID: 20, Chunk IDs: []
Question ID: 21, Chunk IDs: []
Question ID: 22, Chunk IDs: []
Question ID: 23, Chunk IDs: []
Question ID: 24, Chunk IDs: []
Question ID: 25, Chunk IDs: []
Question ID: 26, Chunk IDs: []
Question ID: 27, Chunk IDs: []
Question ID: 28, Chunk IDs: []
Question ID: 29, Chunk IDs: []
Batch 2
Question ID: 0, Chunk IDs: [

In [91]:

questions_container = {i: [] for i in range(30)}

In [92]:
questions_container

{0: [],
 1: [],
 2: [],
 3: [],
 4: [],
 5: [],
 6: [],
 7: [],
 8: [],
 9: [],
 10: [],
 11: [],
 12: [],
 13: [],
 14: [],
 15: [],
 16: [],
 17: [],
 18: [],
 19: [],
 20: [],
 21: [],
 22: [],
 23: [],
 24: [],
 25: [],
 26: [],
 27: [],
 28: [],
 29: []}

In [95]:
len(answer_extended_chunks)

19

In [99]:
for i, data in enumerate(answer_extended_chunks):
    print(f"Batch {i+1}")
    for item in data:
        #print(f"----> item.question_id: {item.question_id}")
        #print(questions_container[item.question_id])
        questions_container[item.question_id].extend(item.chunk_ids)

Batch 1
Batch 2
Batch 3
Batch 4
Batch 5
Batch 6
Batch 7
Batch 8
Batch 9
Batch 10
Batch 11
Batch 12
Batch 13
Batch 14
Batch 15
Batch 16
Batch 17
Batch 18
Batch 19


In [100]:
questions_container

{0: [],
 1: [],
 2: [],
 3: ['B094FM9L6Q', 'B07PC7JKSB'],
 4: [],
 5: ['B098RZX5ZL',
  'B07W88Q6TR',
  'B08MQ93D6Q',
  'B0C73DDC7Q',
  'B0CBS4FG4G',
  'B09158WPGB'],
 6: ['B00QVRTUQ6'],
 7: [],
 8: ['B074VLNLGZ'],
 9: ['B0C2W35TDM', 'B0B24PLBRF'],
 10: ['B07ZVHBY7Z', 'B0888BDZS2', 'B08BF72475', 'B085CCMX84', 'B08136P6Q2'],
 11: [],
 12: [],
 13: [],
 14: [],
 15: [],
 16: ['B07TV4FTKS', 'B07919L929'],
 17: [],
 18: ['B013R9SBOC'],
 19: ['B06XPFXF8B',
  'B01KOE8D9Q',
  'B07VF75H1P',
  'B09FZC7BHK',
  'B07P2Z8P5S',
  'B08H5NYYRF',
  'B09G9YBPG5'],
 20: [],
 21: [],
 22: [],
 23: [],
 24: [],
 25: [],
 26: [],
 27: ['B016DGF0ZY'],
 28: ['B01M9FYRQ7'],
 29: []}

In [101]:
for i, data in enumerate(response.synthetic_questions):
    questions_container[i].extend(data.chunk_ids)

In [102]:
questions_container

{0: ['B0B5LRMSQX'],
 1: ['B0BSLCXVY4'],
 2: ['B07ZM19Y9H'],
 3: ['B094FM9L6Q', 'B07PC7JKSB', 'B091JVYV8F'],
 4: ['B08H2323M6'],
 5: ['B098RZX5ZL',
  'B07W88Q6TR',
  'B08MQ93D6Q',
  'B0C73DDC7Q',
  'B0CBS4FG4G',
  'B09158WPGB',
  'B01F9RGSFE'],
 6: ['B00QVRTUQ6', 'B078P57ZWL'],
 7: ['B09F9KF1L8'],
 8: ['B074VLNLGZ', 'B09DBRM3R5'],
 9: ['B0C2W35TDM', 'B0B24PLBRF', 'B07P85CDHG'],
 10: ['B07ZVHBY7Z',
  'B0888BDZS2',
  'B08BF72475',
  'B085CCMX84',
  'B08136P6Q2',
  'B07PWTYHPL'],
 11: ['B09M3P2W7M'],
 12: ['B006C1ILUC'],
 13: ['B08DXTLCBC'],
 14: ['B091J2D4TG'],
 15: ['B078XQ9LND', 'B07P85CDHG'],
 16: ['B07TV4FTKS', 'B07919L929', 'B07J3X1DPG', 'B07J6PW4SH'],
 17: ['B08H2323M6', 'B08DXTLCBC'],
 18: ['B013R9SBOC', 'B07WQFSPR1', 'B07BKPQMQL'],
 19: ['B06XPFXF8B',
  'B01KOE8D9Q',
  'B07VF75H1P',
  'B09FZC7BHK',
  'B07P2Z8P5S',
  'B08H5NYYRF',
  'B09G9YBPG5',
  'B00XLIG4GW',
  'B091JVYV8F'],
 20: ['B08D69CDSX', 'B01F9RGSFE'],
 21: ['B07GDJ3662', 'B09P197SM7'],
 22: ['B0798JJ2T5', 'B0B5ZP46BJ'],

In [103]:
def get_description(parent_asin: str) -> str:
    
    points = qdrant_client.scroll(
        collection_name=collection_name,
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        )
    )[0]

    return points[0].payload["preprocessed_description"]

In [104]:
get_description('B08DXTLCBC')

'WiFi 6 Card PCIe 3000Mbps, QGOO Wireless Adapter with Bluetooth 5.0 Dongle for PC AX200 Pro WiFi Adapter Express Network Dual Band 6dBi High Gain Antenna (Only Support Desktop for Windows10 64bit)SPEED MULTIPLIED:Compared with the previous generation of wireless technology, AX200 PRO supports 160MHz broadband and OFDMA technology. With the support of the new WI-FI6 technology, the speed has been digitally increased, and it can easily cope with high-speed transmission of large games, 4K audio and video, and VR. Dual Band 2.42GHz/574Mbps 5.8GHz/2400Mbps 5dBi High Gain Antenna. Don’t be afraid of wall obstacles and other blocking signals. (3000 Mbps is the theoretical speed) TRUE LOVE OF GAMING PLAYERS:Using a new generation of MU-MIMO multi-user multi-input multi-output technology, multiple users and multiple devices can simultaneously connect to the Internet without queuing. Team battles must be done without delay. The prestigious Inter AX200 pro chip has excellent performance in large

In [105]:
def generate_golden_answer(question, relevant_chunks):

    class GoldenAnswer(BaseModel):
        answer: str = Field(description="The ideal reference answer for the question.")

    PROMPT = f"""
You are the shopping assistant of an e-shop. You are also being used to generate the ideal reference answer for a single customer question, to serve as ground truth for evaluating a RAG system.

## Context
A customer asked a question. A retrieval system has already gathered the products most relevant to it. Your job is to write the best possible answer a helpful shopping assistant could give, using ONLY those products.

## Inputs
- `question`: the customer's question.
- `products`: a list of relevant products, each with a `text` description. This may be empty or may not actually contain the answer.

## How to write the answer
- Ground every factual claim strictly in the provided product descriptions. Never invent products, prices, specs, availability, or details that are not present in the text.
- Answer in the voice of a friendly, concise shopping assistant speaking directly to the customer.
- Use only the information needed to answer well; don't dump every product if only some are relevant.
- If the products only partially answer the question, answer what you can and clearly state what information isn't available.
- If the products do not contain the information needed to answer at all (or the list is empty), do not guess. Respond the way a good assistant would when the shop doesn't carry the item or the info isn't available (e.g. politely say you couldn't find it), and do not fabricate an answer.

## Question
{json.dumps(question, indent=2)}

## Products
{json.dumps(relevant_chunks, indent=2)}"""

    response, raw_response = instructor_client.create_with_completion(
        messages=[
            {"role": "system", "content": PROMPT}
        ],
        reasoning={"effort": "none"},
        response_model=GoldenAnswer
    )

    return response.answer

In [106]:
reference_dataset = []

for i, data in enumerate(response.synthetic_questions):

    print(f"Working on question {i+1}")

    relevant_chunks = [get_description(item) for item in questions_container[i]]

    golden_answer = generate_golden_answer(data.question, relevant_chunks)

    reference_dataset.append({
        "question": data.question,
        "ground_truth": golden_answer,
        "reference_context_ids": questions_container[i],
        "reference_descriptions": relevant_chunks
    })

Working on question 1
Working on question 2
Working on question 3
Working on question 4
Working on question 5
Working on question 6
Working on question 7
Working on question 8
Working on question 9
Working on question 10
Working on question 11
Working on question 12
Working on question 13
Working on question 14
Working on question 15
Working on question 16
Working on question 17
Working on question 18
Working on question 19
Working on question 20
Working on question 21
Working on question 22
Working on question 23
Working on question 24
Working on question 25
Working on question 26
Working on question 27
Working on question 28
Working on question 29
Working on question 30


In [107]:
reference_dataset

[{'question': 'Do you have a 75-inch Sony TV with Dolby Vision and a native 120Hz refresh rate for PS5 gaming?',
  'ground_truth': 'Yes — we do have a 75-inch Sony TV that matches that. The Sony X85K Series 75-inch (KD75X85K) includes Dolby Vision HDR and a native 120Hz refresh rate. It also has PS5-focused gaming features, including HDMI 2.1 support with 4K/120, VRR, and ALLM, plus features designed to enhance gaming picture quality for PlayStation 5.',
  'reference_context_ids': ['B0B5LRMSQX'],
  'reference_descriptions': ['Sony 75 Inch 4K Ultra HD TV X85K Series: LED Smart Google TV with Dolby Vision HDR and Native 120HZ Refresh Rate KD75X85K- Latest ModelINTELLIGENT TV PROCESSING– The 4K HDR Processor X1 delivers a picture that is smooth and clear, full of rich colors and detailed contrast. INTELLIGENT MOTION HANDLING – See blur-free picture quality in fast-moving sports and action-packed movies with native 120Hz refresh rate and Motionflow XR technology. WIDE SPECTRUM OF COLORS- R

In [108]:
ls_client = Client()

In [109]:
dataset_name = "rag-evaluation-dataset-extended"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="RAG evaluation dataset for 1000 items"
)

In [110]:
for item in reference_dataset:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"]
        },
        outputs={
            "ground_truth": item["ground_truth"],
            "reference_context_ids": item["reference_context_ids"],
            "reference_descriptions": item["reference_descriptions"]
        }
    )